In [ ]:
from typing import Annotated, List, TypedDict
from collections.abc import Sequence

from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_deepseek import ChatDeepSeek

from operator import add
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 構造化出力用の Schema
class Section(BaseModel):
    name: str = Field(
        description="レポートの章タイトル",
    )
    description: str = Field(
        description="本章のテーマと核心的な内容の概要",
    )

class Sections(BaseModel):
    sections: List[Section] = Field(
        description="レポートの章のリスト",
    )

# オーケストレーター
planner = model.with_structured_output(Sections)

# グラフの状態
class OverAllState(TypedDict):
    topic: str  # レポートのテーマ
    sections: list[Section]  # レポートの章のリスト
    completed_sections: Annotated[
        list, add
    ]  # すべてのワーカーノードがこのフィールドに並行して書き込む
    final_report: str  # 最終レポート


# ワーカーノードの状態
class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, add]

# ノード
def orchestrator(state: OverAllState) -> OverAllState:
    """オーケストレーター：レポート作成計画を生成する"""

    # レポートの章立てを生成
    report_sections = planner.invoke(
        [
            SystemMessage(content="このレポートの章立てを作成してください。"),
            HumanMessage(content=f"レポートのテーマは以下の通りです：{state['topic']}"),
        ]
    )

    return {"sections": report_sections.sections}

def model_call(state: WorkerState) -> WorkerState:
    """ワーカーノード：レポート内の1つの章を執筆する"""

    # 章の内容を生成
    section = model.invoke(
        [
            SystemMessage(
                content=(
                    "提供された章タイトルと章の説明に基づいてレポート内容を作成してください。"
                    "各章の前に余計な導入文を追加しないでください。"
                    "Markdown 形式を使用してください。"
                )
            ),
            HumanMessage(
                content=(
                    f"章タイトル：{state['section'].name}\n"
                    f"章の説明：{state['section'].description}"
                )
            ),
        ]
    )

    # 生成した章を完成済み章のリストに書き込む
    return {"completed_sections": [section.content]}


def synthesizer(state: OverAllState) -> OverAllState:
    """すべての章を統合して完全なレポートにする"""

    # すべての完成済み章を取得
    completed_sections = state["completed_sections"]

    # 完成済み章を連結して最終レポートにする
    completed_report_sections = "\n\n---\n\n".join(completed_sections)

    return {"final_report": completed_report_sections}


# 条件付きエッジ関数：計画内の各章について model_call ワーカーノードを作成する
def assign_workers(state: OverAllState) -> Sequence[Send]:
    """計画内の各章にワーカーノードを1つずつ割り当てる"""

    # Send API を使って各章の執筆タスクを並行して起動する
    return [Send("model_call", {"section": s}) for s in state["sections"]]


# ワークフローを構築
builder = StateGraph(state_schema=OverAllState)

# ノードを追加
builder.add_node("orchestrator", orchestrator)
builder.add_node("model_call", model_call)
builder.add_node("synthesizer", synthesizer)

# エッジを追加して各ノードを接続
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges(
    "orchestrator",
    assign_workers,
    ["model_call"],
)
builder.add_edge("model_call", "synthesizer")
builder.add_edge("synthesizer", END)

# ワークフローをコンパイル
graph = builder.compile()

# ワークフローを呼び出す
state = graph.invoke(
    {"topic": "大規模言語モデルのスケーリング則に関するレポートを作成してください"}
)

from IPython.display import Markdown, display

# ワークフローグラフを表示
display(graph)
# レポートをレンダリング
Markdown(state["final_report"])